In [2]:
#1. Redes Semánticas (NetworkX)
import networkx as nx

G = nx.DiGraph()

# Nodos (conceptos)
conceptos = ["Animal", "Mamifero", "Ave", "Perro", "Gato", "Canario"]
G.add_nodes_from(conceptos)

# Relaciones jerárquicas
G.add_edge("Mamifero", "Animal", relation="es_un")
G.add_edge("Ave", "Animal", relation="es_un")
G.add_edge("Perro", "Mamifero", relation="es_un")
G.add_edge("Gato", "Mamifero", relation="es_un")
G.add_edge("Canario", "Ave", relation="es_un")

# Relaciones adicionales
G.add_edge("Perro", "Gato", relation="enemigo_de")
G.add_edge("Canario", "Gato", relation="presa_de")

# Consultas
def obtener_superclases(concepto):
    return [v for _, v, d in G.out_edges(concepto, data=True) if d["relation"] == "es_un"]

def obtener_relaciones(concepto):
    return [(v, d["relation"]) for _, v, d in G.out_edges(concepto, data=True)]

print("Superclases de Perro:", obtener_superclases("Perro"))
print("Relaciones de Gato:", obtener_relaciones("Gato"))

Superclases de Perro: ['Mamifero']
Relaciones de Gato: [('Mamifero', 'es_un')]


In [7]:
2#Ontologías OWL
from owlready2 import *

# 1. Limpiar memoria previa
default_world.ontologies.clear()

# 2. Definir la ontología
onto = get_ontology("http://ejemplo.org/mascotas.owl")

with onto:
    class Animal(Thing): pass
    class Perro(Animal): pass
    class Persona(Thing): pass
    class tieneMascota(Persona >> Animal): pass
    class ladra(Animal >> bool): pass

# 3. Crear las instancias
juan = Persona("Juan")
firulais = Perro("Firulais")

# 4. Asignar relaciones
juan.tieneMascota = [firulais]
firulais.ladra = [True]

# 5. Para evitar el error de Java/HermiT, comentamos la sincronización del razonador:
# sync_reasoner() 

# 6. Consultas e impresión de resultados
for mascota in juan.tieneMascota:
    print("Juan tiene mascota:", mascota.name)

print("¿Firulais ladra?:", firulais.ladra[0])

Juan tiene mascota: Firulais
¿Firulais ladra?: True


In [8]:
3# Logica posicional 
# Base de hechos inicial
hechos = {
    "llueve": True,
    "paraguas": False
}

# Reglas (premisas → conclusión)
reglas = [
    (["llueve", "not paraguas"], "mojarse"),
    (["llueve"], "suelo_mojado")
]

def evaluar_literal(literal, hechos):
    if literal.startswith("not "):
        return not hechos.get(literal[4:], False)
    return hechos.get(literal, False)

def motor_inferencia(hechos, reglas):
    nuevos = True
    while nuevos:
        nuevos = False
        for premisas, conclusion in reglas:
            if conclusion not in hechos:
                if all(evaluar_literal(p, hechos) for p in premisas):
                    hechos[conclusion] = True
                    nuevos = True
    return hechos

resultado = motor_inferencia(hechos, reglas)
print("Hechos inferidos:", resultado)


Hechos inferidos: {'llueve': True, 'paraguas': False, 'mojarse': True, 'suelo_mojado': True}


In [12]:
#4. Lógica de Primer Orden (Kanren)
from kanren import Relation, facts, var, vars, run

# 1. Definir la relación de parentesco
padre = Relation()

# 2. Agregar los hechos a la base de conocimiento
facts(padre,
      ("Juan", "Pedro"),
      ("Juan", "Ana"),
      ("Pedro", "Luis"))

# 3. Consulta: Obtener los hijos de una persona
def obtener_hijos(persona):
    x = var()
    return run(10, x, padre(persona, x))

# 4. Consulta: Obtener los abuelos de una persona (CORREGIDO)
def obtener_abuelo(nieto):
    # Usamos vars(2) para generar dos variables lógicas de forma iterable
    x, y = vars(2)
    return run(10, x, padre(x, y), padre(y, nieto))

# 5. Ejecutar e imprimir las consultas
print("Hijos de Juan:", obtener_hijos("Juan"))
print("Abuelos de Luis:", obtener_abuelo("Luis"))

Hijos de Juan: ('Ana', 'Pedro')
Abuelos de Luis: ('Juan',)


In [20]:
#5. Marcos / Frames (POO)
class Frame:
    def __init__(self, name, parent=None, **slots):
        self.name = name
        self.parent = parent
        self.slots = slots

    def obtener(self, slot):
        if slot in self.slots:
            return self.slots[slot]
        if self.parent:
            return self.parent.obtener(slot)
        return None

# Definición de la jerarquía
animal = Frame("Animal", None, tiene_sangre=True)
perro = Frame("Perro", animal, sonido="ladrar")
firulais = Frame("Firulais", perro, color="café")

print("Firulais tiene sangre:", firulais.obtener("tiene_sangre"))
print("Sonido de Firulais:", firulais.obtener("sonido"))

Firulais tiene sangre: True
Sonido de Firulais: ladrar


In [21]:
#6. Sistema de Reglas de Producción
hechos = {"temperatura": 39, "tos": True, "dolor": False}

reglas = [
    (lambda h: h["temperatura"] > 38 and h["tos"], lambda h: h.update({"diagnóstico": "Fiebre con tos"})),
    (lambda h: h["temperatura"] > 38 and not h["tos"], lambda h: h.update({"diagnóstico": "Fiebre simple"})),
    (lambda h: h["dolor"], lambda h: h.update({"recomendación": "Analgésico"}))
]

for cond, accion in reglas:
    if cond(hechos):
        accion(hechos)

print("Estado final de hechos:", hechos)

Estado final de hechos: {'temperatura': 39, 'tos': True, 'dolor': False, 'diagnóstico': 'Fiebre con tos'}


In [22]:
# =====================================================================
# EJEMPLO COMPLEJO: Sistema de Diagnóstico Mecánico basado en Marcos
# =====================================================================

class FrameVehiculo:
    def __init__(self, nombre, padre=None, **slots):
        self.nombre = nombre
        self.padre = padre
        self.slots = slots  # Ranuras (atributos/propiedades)

    def obtener_slot(self, slot):
        """Inferencia por herencia: busca localmente o sube en la jerarquía"""
        if slot in self.slots:
            return self.slots[slot]
        if self.padre:
            return self.padre.obtener_slot(slot)
        return "Atributo no definido"

# 1. Marco General (Estereotipo raíz)
vehiculo = FrameVehiculo(
    "VehiculoGeneral", 
    tiene_motor=True, 
    requiere_mantenimiento=True,
    tipo_revision="Inspección General"
)

# 2. Submarcos por Tipo de Combustible (Heredan de VehiculoGeneral)
auto_gasolina = FrameVehiculo("AutoGasolina", vehiculo, tipo_combustible="Gasolina", tiene_bujias=True)
auto_electrico = FrameVehiculo("AutoElectrico", vehiculo, tipo_combustible="Batería Li-Ion", tiene_bujias=False)

# 3. Submarco de una falla específica (Hereda de AutoGasolina)
falla_arranque = FrameVehiculo(
    "FallaArranqueGasolina", 
    auto_gasolina, 
    componente_critico="Batería / Marcha",
    sintoma_principal="El motor no da marcha al girar la llave",
    solucion_recomendada="Verificar voltaje de batería o reemplazar marcha"
)

# --- PRUEBAS Y CONSULTAS DE INFERENCIA ---
print("=== DIAGNÓSTICO AVANZADO CON MARCOS ===")
print("Falla analizada:", falla_arranque.nombre)
print("-> Síntoma:", falla_arranque.obtener_slot("sintoma_principal"))          # Propio de la falla
print("-> Solución:", falla_arranque.obtener_slot("solucion_recomendada"))      # Propio de la falla
print("-> ¿Tiene bujías?:", falla_arranque.obtener_slot("tiene_bujias"))         # Heredado de AutoGasolina
print("-> ¿Tiene motor?:", falla_arranque.obtener_slot("tiene_motor"))           # Heredado de VehiculoGeneral
print("-> Tipo de revisión:", falla_arranque.obtener_slot("tipo_revision"))     # Heredado de VehiculoGeneral

=== DIAGNÓSTICO AVANZADO CON MARCOS ===
Falla analizada: FallaArranqueGasolina
-> Síntoma: El motor no da marcha al girar la llave
-> Solución: Verificar voltaje de batería o reemplazar marcha
-> ¿Tiene bujías?: True
-> ¿Tiene motor?: True
-> Tipo de revisión: Inspección General
